# Run Models #

This JupyterNotebook file will run FBA on the models.

**Functions:**

1 - products2FBA(substrates, SUR, O2FluxLB, objective, inputDirectory_models,inputDirectory_general)

2 - substrates2FBA(product, substrates, SUR, O2FluxLB, objective, inputDirectory_general, inputDirectory_models)

3 - runFBA(product, substrate, pathway, SUR, O2FluxLB, objectiveSet, inputDirectory_general, inputDirectory_models)

4 - setModel(model, pathwayRxnsDF, substrate, SUR)

5 - productInfo(inputDirectory_models, inputDirectory_general)

6 - substratePWinfo(substrates, inputDirectory_general)

**Requirements:**
- Must specify the product and substrate info document names in the productInfo() and substratePWinfo() functions themselves (path to the containing folder is passed in though). This could easily be modified...
- Product and substrate uptake PW Excel files must use the required format (same as used for the SPI analysis)
- Models must have correct naming convention:
    - 'model_GS_'+productName+'.xml'; the productName in the model must match the productName in the product info excel file
    


#### In Future: ####
- May want to make input and output paths global variables, or figure out how to manage them with objects?

** *Run model log at bottom of document*

In [ ]:
import cobra
import cameo
import math
import escher
import plotly
import os

import numpy as np
from scipy import stats
import pandas as pd
import sympy as sy
from datetime import date, datetime
import time
import glob #For createModels() function

date = datetime.strftime(datetime.now(), '%Y-%m-%d')

In [ ]:
inputDirectory_general = os.path.abspath("RunModels/Input/") + os.path.sep
inputDirectory_models = os.path.abspath("RunModels/Input/XMLfiles/") + os.path.sep

outputDirectory_general = os.path.abspath("RunModels/Output/") + os.path.sep
outputDirectory_FBA = os.path.abspath("RunModels/Output/FBA/") + os.path.sep

In [ ]:
#Main - Run models#
#Run code above and functions below before running Main cells#

#Uncomment/create new substrate lists of desired targetted intermediates

#substrates = ['Acetaldehyde', 'Acetate', 'EG_Glycolate', 'EG_SACA', 
#              'Ethanol', 'Formate_Formolase','Formate_RGP-Serine', 'Formate_SerineCycle',
#              'Glucose', 'Glycerol', 'Glycolaldehyde_Glycolate', 'Glycolaldehyde_SACA',
#              'Methanol_Formolase', 'Methanol_RuMP', 'Methanol_SerineCycle',
#              'Xylose'
#             ] 
#substrates = ['Acetate', 'Ethanol', 'EG_SACA', 'EG_Glycolate', 'Glucose']
#substrates = ['CO_1:H2_0', 'CO_1:H2_0.5', 'CO_1:H2_1', 'CO_1:H2_2', 
#             'FOR_1:H2_0', 'FOR_1:H2_0.5', 'FOR_1:H2_1', 'FOR_1:H2_2']
substrates = ['Methanol_MethylTransferase_1','Methanol_MethylTransferase_0.5','Methanol_MethylTransferase_0.1','Methanol_MethylTransferase_0']
#substrates = ['CO_2:CO1_4']
SUR = -10
O2FluxLB = -20
objective = 'Product' #'Product', 'Biomass_iJO1366' for iJO1366 model or 'Biomass_iHN637' for iHN637 model, or the ID for the desired reaction
modelID = 'iHN637'

tic = time.perf_counter()
products2FBA(substrates, SUR, O2FluxLB, objective, inputDirectory_models,inputDirectory_general)
toc = time.perf_counter()
print(f"Models executed in {toc - tic:0.4f} seconds")

In [ ]:
## Main - Quick Test ##

#display(productInfo(inputDirectory_models,inputDirectory_general))
modelName = 'model_GS_Acetyl.xml'

#Read in model:
model_initial = cameo.load_model(inputDirectory_models+str(modelName))

#display(model_initial.reactions.get_by_id('ATPM').lower_bound)
substrate = 'Methanol_RuMP' #'FOR_1:H2_0' #'Formate_RGP-Serine'
modelID = 'iJO1366'
pathwayRxnsDF = substratePWinfo(inputDirectory_general)
SUR = -10
#[model, substrateExchange, S_MW] = setiHN637Model(model_initial, pathwayRxnsDF, substrate, SUR)
[model, substrateExchange, S_MW] = setModel(model_initial, pathwayRxnsDF, substrate, SUR) #For iJO1366

#display(model)
#display(model.reactions.get_by_id(substrateExchange))
#display(model.reactions.get_by_id('EX_h2_e'))
#display(model.reactions.get_by_id('EX_fru_e'))

#display(model.reactions.get_by_id('EX_r13bdo_e'))

model.objective = 'EX_acetyl_e' #EX_product_e, biomass, or another specified reaction ID, passed in
#model.objective = 'EX_lys__L_e'
fba_result = cameo.fba(model)
display(model.summary())



In [ ]:
## Main - Quick Test - May 7, 2021 ##

#display(cameo.fba(model).fluxes['EX_pyr_e'])
#display(model.metabolites.pyr_c.summary())
#display(model.metabolites.accoa_c.summary())

#display(model.reactions.get_by_id('SGATc').summary())




display(model.metabolites.get_by_id('g3p_c').summary())
#display(model.metabolites.get_by_id('datp_c').summary())
#display(model.metabolites.get_by_id('dadp_c').summary())
#display(model.metabolites.get_by_id('damp_c').summary())
#display(model.metabolites.get_by_id('dad_2_c').summary())
#display(model.metabolites.get_by_id('2dr1p_c').summary())
#display(model.metabolites.get_by_id('2dr5p_c').summary())
#display(model.metabolites.get_by_id('acald_c').summary())
#display(model.metabolites.get_by_id('accoa_c').summary())

#display(model.metabolites.get_by_id('r5p_c').summary())
#display(model.metabolites.get_by_id('r1p_c').summary())
#display(model.metabolites.get_by_id('adn_c').summary())


In [ ]:
def products2FBA(substrates, SUR, O2FluxLB, objective, inputDirectory_models,inputDirectory_general):
#Loops through all products/models read in via productInfo() and calls on substrates2FBA() 
#to execute FBA for all substrates, for the given product.
#Combines results summary DF from each product together to build one large results summary DF 
#containing FBA results for all products and substrates.
#Overall FBA summary DF is returned.

    #Import product info from CSV file
    productsDF = productInfo(inputDirectory_models,inputDirectory_general)
    display(productsDF) #Check
    
    #Import substrate assimilation pathways from Excel file as dataframe. DF is passed to substrates2FBA() function upon each product iteration
    pathwayRxnsDF = substratePWinfo(inputDirectory_general)
    #Check: #display(pathwayRxnsDF)
    
    #Loop through productsDF and convert each row to a list (for each product), then pass this to the substrates2FBA() function
    for index, row in productsDF.iterrows():
        product =  productsDF.loc[index]
        #print('Current product df:') #Check
        #display(product) #Check
        #display(product.ExchangeRxn)
        print('Currently working on product:', product['ProductName']) #Display which product is currently being worked on
        print('On substrate:')
        
        #Passes product, substrates and other required variables to substrates2FBA()
        df = substrates2FBA(product, substrates, pathwayRxnsDF, SUR, O2FluxLB, objective, inputDirectory_general, inputDirectory_models)     
        
        if index == 0:
            df_new = df
        elif index >0:
            frames = [df_new, df]
            df_new = pd.concat(frames)
    
    df_new.to_csv(outputDirectory_FBA + 'Summaries/' + 'OverallSummary_' + date + '.csv') #To CSV file
    df_new.to_html(outputDirectory_FBA + 'Summaries/'  + 'OverallSummary' + date + '.html') #To HTML file
    print('Analysis is complete.')
    return(df_new)

#Check:
#substrates = ['Acetate', 'Ethanol', 'EG_saca', 'EG_glycolate', 'Glucose']
#SUR = -10
#O2FluxLB = -20
#objective = 'Product' #'Product', 'Biomass', or the ID for the desired reaction
#products2FBA(substrates, SUR, O2FluxLB, objective, inputDirectory_models,inputDirectory_general)

In [ ]:
def substrates2FBA(product, substrates, pathwayRxnsDF, SUR, O2FluxLB, objective, inputDirectory_general, inputDirectory_models):
#For specified product/model, loops through substrates calling on runFBA()
#Combines specified FBA results from runFBA() as one big array and then converts to DF and calculates yields
#Returns df containing results for single product and all substrates 

    productName = product['ProductName']
    P_MW = product['MW_BP']
    display(P_MW)
    #Set objective:
    if objective == 'Product': #If user specifies "Product" as the objective, set the product exchange flux as the objective
        objectiveSet = product['ExchangeRxn'] 
    elif objective == 'Biomass_iJO1366': #Else if the user specifies "Biomass" as the objective, set biomass as the objective
        objectiveSet = 'BIOMASS_Ec_iJO1366_WT_53p95M' #This is for the iJO1366 model only!
    elif objective == 'Biomass_iHN637':
        objectiveSet = 'BIOMASS_Cl_DSM_WT_46p666M1'
    else:
        objectiveSet = objective #Otherwise, use the reaction provided as the objective
    #Check: #print('Objective:', objective) #print('ObjectiveSet:', objectiveSet)
    
    #i=0
    resultsList = [] #Create empty list to store results
    for s in substrates: #loops through all substrates for analysis
    #for index, row in pathwayRxnsDF.iterrows(): #Loops through all rows of assimilation pathways dataframe
        #pathway = pathwayRxnsDF.loc[index] #Pass row for substrate indicated by index
        display(s) #Display which substrate is being worked on
        FBAresults = runFBA(product, s, pathwayRxnsDF, SUR, O2FluxLB, objectiveSet, inputDirectory_general, inputDirectory_models) #Runs FBA and returns summary list (vector)
        resultsList.append(FBAresults) #Adds summary list to growing list of FBA results (list of lists...)
        #display(pathway) #Check
        #i=i+1

    #Convert list of results to dataframe:    
    df = pd.DataFrame(resultsList)
    df = df.rename(index=str, columns={0: 'Date', 1:'Substrate', 2:'Product', 3:'Objective', 4:'S_ExchangeLB', 5:'O2_ExchangeLB', 6:'H2_ExchangeLB', 7:'ATPM_LB', 8:'S_Flux', 9:'O2_Flux', 10: 'H2_Flux', 11:'CO2_Flux', 12:'Obj_Flux', 13:'S_MW', 14:'By-Products'})
    
    #Calculate yields (molar and mass)
    df['Yield_Molar'] = abs(df['Obj_Flux']/df['S_Flux']) #Product molar yield (for biomass this is gDW/mmol for other products ids mmol/mmol)
    

    if objective == 'Product':
        df['Yield_Mass'] = df['Yield_Molar']*P_MW/df['S_MW'] #Product mass yield 
    elif objective == 'Biomass_iJO1366' or 'Biomass_iHN637':
        df['Yield_Mass'] = df['Yield_Molar']*(10**3)/df['S_MW'] #Biomass mass yield
    else: df['Yield_Mass'] = np.nan #User will have to calculate yield for other objectives
    
    df = df.drop(columns=['S_MW']) #Only needed this column for calculation, can remove for export
    df.to_csv(outputDirectory_FBA + 'Summaries/' + 'FBA_p' + productName  + '_ResultsSummary' + '.csv') #To CSV file
    df.to_html(outputDirectory_FBA + 'Summaries/'  + 'FBA_p' + productName  + '_ResultsSummary' + '.html') #To HTML file
    #display(df) #Check
    
    
    return(df)

#Check:
#product = ['Farnesene', 'C15H24', 'EX_frene_e', 204.34866, 'model_GS_Farnesene.xml']
#substrates = ['Acetate', 'Ethanol', 'EG_saca', 'EG_glycolate', 'Glucose']
#SUR = -10
#O2FluxLB = -20
#objective = 'Product' #'Product', 'Biomass', or the ID for the desired reaction
#df = substrates2FBA(product, substrates, SUR, O2FluxLB, objective, inputDirectory_general, inputDirectory_models)
#display(df)

In [ ]:
def runFBA(product, substrate, pathwayRxnsDF, SUR, O2FluxLB, objectiveSet, inputDirectory_general, inputDirectory_models):
#Runs single instance of FBA based on the model/product, substrate and pathway
#Before running FBA, all reactions in assimilation pathways are set to appropriate bounds via the setModel() fuction
#Returns 1D array of results and exports FBA results to CSV and HTML
    
    #Set product info:
    productName = product['ProductName']
    productExchangeRxn = product['ExchangeRxn']
    modelName = product['ModelName']
    print('Product Name:', productName, '\nproductExchangeRxn:', productExchangeRxn, '\nmodelName:', modelName ) #Check
    
    #Read in model:
    model_initial = cameo.load_model(inputDirectory_models+str(modelName)) #load from file
    #model = resetModel(model_initial, inputDirectory_general)
    isThreeGas = False # Allows for the iHN637 to take on more complex gas mixtures
    
    #Set model assimilation pathways
    if modelID is 'iHN637' and not(substrate == 'Methanol_MethylTransferase_0' or substrate == 'Methanol_MethylTransferase_0.1' or substrate == 'Methanol_MethylTransferase_0.5' or substrate == 'Methanol_MethylTransferase_1'): #Using Ecoli reader function for new pathway
        if substrate == 'Flue_Gas':
            isThreeGas = True
        modelInfo = setiHN637Model(model_initial, pathwayRxnsDF, substrate, SUR, isThreeGas) #Returns set model, the rxnID for the substrate exchange reaction and the substrate molecular weight
        model = modelInfo[0] #Model has correct carbon source:H2 ratio set
        substrateExchange = modelInfo[1]
        S_MW = modelInfo[2]
        
    else: #Assumes iJO1366 model used or another model compatible with conventional substrate conversion pathways
        modelInfo = setModel(model_initial, pathwayRxnsDF, substrate, SUR) #Returns set model, the rxnID for the substrate exchange reaction and the substrate molecular weight
        model = modelInfo[0] #Model has substrate pathway turned on and all other assimilation pathways set to default settings
        substrateExchange = modelInfo[1]
        S_MW = modelInfo[2]
    
    #print('Checking the exchange reactions:') #Check
    #print('EG: ',model.reactions.get_by_id('EX_eg_e').lower_bound)
    #print('Gcald: ',model.reactions.get_by_id('EX_gcald_e').lower_bound)
    #print('Acetate: ',model.reactions.get_by_id('EX_ac_e').lower_bound)
    #print('Ethanol: ',model.reactions.get_by_id('EX_etoh_e').lower_bound)
    #print('Glycerol: ',model.reactions.get_by_id('EX_glyc_e').lower_bound)
    #print('Glucose: ',model.reactions.get_by_id('EX_glc__D_e').lower_bound)
    
    #Set O2 uptake (but not for iHN637 model)
    if modelID is not 'iHN637':
        model.reactions.EX_o2_e.lower_bound = O2FluxLB
    
    #Run FBA & save results
    model.objective = objectiveSet #EX_product_e, biomass, or another specified reaction ID, passed in
    fba_result = cameo.fba(model) #Perform FBA
    fileName = 'FBAresult_s'+ substrate + '_p'+ productName + '_t'+ objective
    #Save FBA results:
    fba_result.data_frame.to_html(outputDirectory_FBA + fileName + '.html')
    fba_result.data_frame.to_csv(outputDirectory_FBA + fileName + '.csv')
    #Save model summary:
    model.summary().to_frame().to_html(outputDirectory_FBA + fileName + '_SUMMARY' + '.html')
    model.summary().to_frame().to_csv(outputDirectory_FBA + fileName + '_SUMMARY' + '.csv')
    
    #S_MW read in is the molecular weight of the carbon source. For C+H2 mixtures, 
    #need to modify molecular weight to be (g mixture/mol Carbon source) for mass yield calculations
    if modelID is 'iHN637':
        #(mol C_source/mol mix) = (mol C_source)/(mol C_source + mol H2)
        y_Csource = (cameo.fba(model).fluxes[substrateExchange])/(cameo.fba(model).fluxes[substrateExchange] + cameo.fba(model).fluxes['EX_h2_e'])
        
        #(mol H2/mol mix) (mol H2)/(mol C_source + mol H2)
        y_H2 = (cameo.fba(model).fluxes['EX_h2_e'])/(cameo.fba(model).fluxes[substrateExchange] + cameo.fba(model).fluxes['EX_h2_e'])
        
        #g mix/mol mix  = (mol Csource/mol mix)*(g/mol C_source) + (mol H2/mol mix)*(g/mol H2)
        S_MW_mix =  y_Csource*S_MW + y_H2*2.016 #g mix/mol mix
        
        #g mix/mol CO = (g mix/ mol mix)*(mol mix/mol CO) = (S_MW_mix)/(y_Csource)
        S_MW = S_MW_mix/y_Csource
        
        
    results = [0 for x in range(15)]
    results[0] = date
    results[1] = substrate
    results[2] = productName
    results[3] = objectiveSet
    results[4] = SUR
    
    if modelID is 'iHN637': results[5] = 0
    else: results[5] = model.reactions.EX_o2_e.lower_bound
    
    results[6] = model.reactions.EX_h2_e.lower_bound
    results[7] = model.reactions.ATPM.lower_bound
    results[8] = cameo.fba(model).fluxes[substrateExchange] #First item of pathway list is the ID for the substrate exchange reaction
    
    if modelID is 'iHN637': results[9] = 0
    else: results[9] = cameo.fba(model).fluxes['EX_o2_e']

    results[10] = cameo.fba(model).fluxes['EX_h2_e']
    results[11] = cameo.fba(model).fluxes['EX_co2_e']
    results[12] = cameo.fba(model).fluxes[objectiveSet]
    
    results[13] = S_MW
    results[14] = ''
    
    #print('The substrate molecular weight is: ' + str(S_MW)) #Check
    display(results) #Check
    return(results)


In [ ]:
def setiHN637Model(model, pathwayRxnsDF, substrate, SUR, isThreeGas):

    #This model assumes the intermediate is specifically a H2 gas mixture (And requires specific formatting in the SubstrateUptakePW iHN637 Excel file (see README for more details))
    #More generally, use the below setModel function for non X+H2 gas mixtures, which may require manipulation of the above runFBA function if the iHN637 model is desired for other intermediates
    #You can see that the methane fixation (methanogenesis) pathways have already been added into said runRBA function as exceptions to bypass the utilization of this setiHN647Model function
    
    rxn_ids = [reaction.id for reaction in model.reactions] #Make a list of all the rxn_ids in the model
    
    #Only keep substrate for the specific analysis
    pathwayDF = pathwayRxnsDF.loc[substrate,:]
    print('Substrate pathway for analysis:') #Check
    display(pathwayDF) #Check
    
    #Store substrateExchange rxnID to return from function
    temp = pathwayDF[1].split(',')
    substrateExchange = temp[0]
    
    #Loop through each column of pathwayDF (which is a single row, for single substrate)
    i=0
    for reactions in pathwayDF:
        #print('reactions:')
        #display(reactions) #Check
        
        if reactions is np.nan or reactions is 'Stop': #Once last reaction is done, exit for loop (indicated by a NaN value)
            #print('If statement works')#Check
            break
    
        elif i == 0: #First column in pathwayDF is the substrate MW
            S_MW = reactions 
            #print('S_MW=', S_MW) #Check
    
        elif i == 1: #Second column in pathwayDF is CO exchange reaction (SUR controlled via this reaction)
            data = reactions.split(',') #data will have three values: (0) rxnID, (1) bound setting while assimilation patwhway in use, (2) bound setting while not in use
            #For assimilation pathway being used, rxn bounds set based on data[1]
            #Bound settings are either -1: reverse rxn only, 0: knocked out, 1:forward rxn only OR 2: reversible
            #print('data=' + str(data)) #Check
            
            #Set exchange reaction bound:
            if int(data[1]) is -1 and data[0] in rxn_ids: #Exchange reaction bound setting should be -1, and exchange reaction should exist in the model
                print('Set exchange LB to ', int(SUR)) #Check
                model.reactions.get_by_id(data[0]).lower_bound = SUR #Allow substrate uptake via exchange reaction
                model.reactions.get_by_id(data[0]).upper_bound = 1000
            elif not data[0] in rxn_ids:
                print('Error: The following reaction is used in the assimilation pathway but does not exist in the model: ', data[0])
            else: #If bound setting not -1 setting for exchange reaction is wrong; need to check Excel file with assimilation pathways
                print('Error: Exchange bound uncertainty for rxnID ', data[0])
    
        elif i == 2: #Third column in pathwayDF is H2 exchange reaction
            data = reactions.split(',')#data will have three values: (0) rxnID, (1) bound setting while assimilation pathway in use, (2) bound setting while not in use
            #print('data=' + str(data)) #Check
            print('data[1]=' + str(data[1]))#Check
            model.reactions.get_by_id(data[0]).lower_bound = float(data[1])*SUR
                
        elif i > 2:
            if isThreeGas:
                data = reactions.split(',')#data will have three values: (0) rxnID, (1) lower bound, (2) upper bound
                print('data[1]=' + str(data[1]))#Check
                model.reactions.get_by_id(data[0]).lower_bound = float(data[1])*SUR
                model.reactions.get_by_id(data[0]).upper_bound = float(data[2])*SUR
            else:
                if reactions is not math.isnan(reactions):
                    break
                else:
                    print('Extra reaction included beyond C-source and H2 exchange. Will be ignored.')
                    #print('reactions for i>2:') #Check
                    #display(reactions) #Check
        
        #display(model.reactions.get_by_id(data[0])) #Check
        i=i+1 #Counter of the for loop
        #print('i='+ str(i)) Check
        #End of for loop
        
    #print('EX_co_e lower bound:' + str(model.reactions.get_by_id('EX_co_e').lower_bound)) #Check
    #print('EX_h2_e lower bound:' + str(model.reactions.get_by_id('EX_h2_e').lower_bound)) #Check
              
    return(model, substrateExchange, S_MW)

In [ ]:
def setModel(model,pathwayRxnsDF, substrate, SUR):
#This function sets the appropriate assimilation pathway for the indicated model and substrate via two activities
#(i) Resets all other assimilation pathway reactions to default model values (or user-selected values)
#(ii) Opens (sets) assimilation pathway for desired substrate
#The information needed to make these changes to the model are passed via the pathwaysRxnDF (which originates from Excel data)
#Note that the "open assimilation PW" activity must come after the "reset" activity because some of the reactions are shared between pathways
#The SUR (substrate uptake rate) is also required to set the exchange reaction for the substrate of interest
    
    
    rxn_ids = [reaction.id for reaction in model.reactions] #Make a list of all the rxn_ids in the model
    
    #=======================================================
    ##(i)Reset default values for all other assimilation pathways
    
    #Pathways for all other substrates (not being used for FBA run):
    otherRxnsDF = pathwayRxnsDF.drop(substrate) 
    
    #Loop through all rows in unused assimilation pathways dataframe:
    
    for index, row in otherRxnsDF.iterrows(): 
        #tempPathwayDF = otherRxnsDF.loc[index,:] #Single unused assimilation pathway (of this iteration)
        
        i=0 #Reset counter for inner for loop
        #Loop through each column of given row:
        for reactions in row:
            # display(reactions) #Check
            if reactions is np.nan or reactions is 'Stop': #Once last reaction is done, exit for loop (indicated by a NaN value)
                break
    
            #elif i == 0: #First column in row is the substrate MW, don't need to do anything
                #print('I am a float:', reactions) #Check
    
            elif i > 0: #All reactions in the pathway, starting with exchange reaction
                #print('this is reaction at i>0: ', reactions)#Check
                data = reactions.split(',')#data will have three values: (0) rxnID, (1) bound setting while assimilation patwhway in use, (2) bound setting while not in use
                #For unused assimilation pathways, rxn bounds set based on data[2]
                #Bound settings are either -1: reverse rxn only, 0: knocked out, 1:forward rxn only OR 2: reversible
                #display(data[0]) #Check
                
                #All if statements use "and data[0] in rxn_ids" to first check that reaction is in model (to avoid errors)
                if int(data[2]) is -1 and data[0] in rxn_ids: #Set reverse reaction only
                    #print('Set rxn LB to ', -1000) #Check
                    #print('Set rxn UB to ', 0) #Check
                    model.reactions.get_by_id(data[0]).lower_bound = -1000 #Allow flux through reverse reaction
                    model.reactions.get_by_id(data[0]).upper_bound = 0 #No flux through forward reaction
                elif int(data[2]) is 0 and data[0] in rxn_ids: #Knock-out reaction
                    #print('knock out rxn') #Check
                    model.reactions.get_by_id(data[0]).knock_out()
                elif int(data[2]) is 1 and data[0] in rxn_ids: #Set forward reaction only
                    #print('Set rxn LB to ', 0) #Check
                    #print('Set rxn UB to ', 1000) #Check
                    model.reactions.get_by_id(data[0]).lower_bound = 0 #No flux through reverse reaction
                    model.reactions.get_by_id(data[0]).upper_bound = 1000 #Allow flux through forward reaction
                elif int(data[2]) is 2 and data[0] in rxn_ids: #Set reversible reaction
                    #print('Set rxn LB to ', -1000) #Check
                    #print('Set rxn UB to ', 1000) #Check
                    model.reactions.get_by_id(data[0]).lower_bound = -1000 #Allow flux through reverse reaction
                    model.reactions.get_by_id(data[0]).upper_bound = 1000 #Allow flux through forward reaction
                #elif not data[0] in rxn_ids: #For checking purposes
                    #print('The following reaction does not exist in the model', data[0])
                elif data[0] in rxn_ids:#If the bound setting isn't -1,0,1 or 2 there's something wrong, need to check Excel file with assimilation pathways
                    print('Error: reaction bound uncertainty for rxnID ', data[0])
            
            #display(model.reactions.get_by_id(data[0])) #Check for inner for loop
            #End of inner for loop
            i=i+1 #Counter for outer for loop
        #End of outer for loop
    
    
    #=======================================================
    ##(ii)Open assimilation pathway for desired substrate(s)
    
    #Only keep substrate for the specific analysis
    pathwayDF = pathwayRxnsDF.loc[substrate,:] 
    
    #Store substrateExchange rxnID to return from function
    temp = pathwayDF[1].split(',')
    substrateExchange = temp[0]
    
    #Loop through each column of pathwayDF (which is a single row, for single substrate)
    i=0
    for reactions in pathwayDF:
        #display(reactions) #Check
        if reactions is np.nan or reactions is 'Stop': #Once last reaction is done, exit for loop (indicated by a NaN value)
            #print('If statement works')#Check
            break
    
        elif i == 0: #First column in pathwayDF is the substrate MW
            S_MW = reactions 
            #print('S_MW=', S_MW) #Check
    
        elif i == 1: #Second column in pathwayDF is exchange reaction (SUR controlled via this reaction)
            data = reactions.split(',') #data will have three values: (0) rxnID, (1) bound setting while assimilation patwhway in use, (2) bound setting while not in use
            #For assimilation pathway being used, rxn bounds set based on data[1]
            #Bound settings are either -1: reverse rxn only, 0: knocked out, 1:forward rxn only OR 2: reversible
            #display(data[0]) #Check
            
            #Set exchange reaction bound:
            if int(data[1]) is -1 and data[0] in rxn_ids: #Exchange reaction bound setting should be -1, and exchange reaction should exist in the model
                #print('Set exchange LB to ', int(SUR)) #Check
                model.reactions.get_by_id(data[0]).lower_bound = SUR #Allow substrate uptake via exchange reaction
                model.reactions.get_by_id(data[0]).upper_bound = 0
            elif not data[0] in rxn_ids:
                print('Error: The following reaction is used in the assimilation pathway but does not exist in the model: ', data[0])
            else: #If bound setting not -1 setting for exchange reaction is wrong; need to check Excel file with assimilation pathways
                print('Error: Exchange bound uncertainty for rxnID ', data[0])
    
        elif i > 1: #Third column and beyond are all other reactions in the pathway, following the exchange reaction 
            data = reactions.split(',')#data will have three values: (0) rxnID, (1) bound setting while assimilation patwhway in use, (2) bound setting while not in use
            #For assimilation pathway being used, rxn bounds set based on data[1]
            #Bound settings are either -1: reverse rxn only, 0: knocked out, 1:forward rxn only OR 2: reversible
            #display(data[0]) #Check
            
            #All if statements use "and data[0] in rxn_ids" to first check that reaction is in model (to avoid errors)
            if int(data[1]) is -1 and data[0] in rxn_ids: #Set reverse reaction only
                #print('Set rxn LB to ', -1000) #Check
                #print('Set rxn UB to ', 0) #Check
                model.reactions.get_by_id(data[0]).lower_bound = -1000 #Allow flux through reverse reaction
                model.reactions.get_by_id(data[0]).upper_bound = 0 #No flux through forward reaction
            elif int(data[1]) is 0 and data[0] in rxn_ids: #Knock-out reaction
                #print('knock out rxn') #Check
                model.reactions.get_by_id(data[0]).knock_out()
            elif int(data[1]) is 1 and data[0] in rxn_ids: #Set forward reaction only
                #print('Set rxn LB to ', 0) #Check
                #print('Set rxn UB to ', 1000) #Check
                model.reactions.get_by_id(data[0]).lower_bound = 0 #No flux through reverse reaction
                model.reactions.get_by_id(data[0]).upper_bound = 1000 #Allow flux through forward reaction
            elif int(data[1]) is 2 and data[0] in rxn_ids: #Set reversible reaction
                #print('Set rxn LB to ', -1000) #Check
                #print('Set rxn UB to ', 1000) #Check
                model.reactions.get_by_id(data[0]).lower_bound = -1000 #Allow flux through reverse reaction
                model.reactions.get_by_id(data[0]).upper_bound = 1000 #Allow flux through forward reaction
            elif not data[0] in rxn_ids:
                print('Error: The following reaction is used in the assimilation pathway but does not exist in the model: ', data[0])
            else: #If the bound setting isn't -1,0,1 or 2 there's something wrong, need to check Excel file with assimilation pathways
                print('Error: reaction bound uncertainty for rxnID ', data[0])
        
        #display(model.reactions.get_by_id(data[0])) #Check
        i=i+1 #Counter of the for loop
    #End of for loop
    
    display(model)#Check
    display(substrateExchange)#Check
    display(S_MW)#Check
    return(model, substrateExchange, S_MW)

In [ ]:
##These two functions carry out data input from CSV files 
##(import product, substrate, model, and assimilation pathway data)

def productInfo(inputDirectory_models, inputDirectory_general):
#This function reads the names of all the product models in the modelDirectory folder, 
#and reads the product info from a CSV file in the bioproductInfoDirectory folder.
#This info is combined in the returned product info dataframe

    ##Read models present in input folder and organize into dataframe with product name and model name
    import os
    i=0
    file_count = len(glob.glob(inputDirectory_models + '*.xml'))#Only counts files of type .xml
    array = [[0 for x in range(2)] for y in range(int(file_count))] #x_columns, y_rows
    #display(file_count) #Check
    for filename in glob.glob(inputDirectory_models + '*.xml'):
        filename2 = filename.replace(inputDirectory_models,'') #Remove folder directory from filename
        
        if modelID is 'iHN637':
            productName1 = filename2.replace('iHN637_', '') #Remove model pre-fix from product name
            productName2 = productName1.replace('.xml', '') #Remove file type from product name
        else: #Assumes conventional model naming otherwise
            productName1 = filename2.replace('model_GS_', '') #Remove model pre-fix from product name
            productName2 = productName1.replace('.xml', '') #Remove file type from product name
        #display(filename) #Check
        #display(productName2) #Check
            
        #Create array with product name and filename
        array[i][0] = productName2 #table[column][row]
        array[i][1] = filename2
        i = i+1
    #Convert array of product + file names to dataframe
    df = pd.DataFrame(array)
    df = df.rename(index=str, columns={0: 'ProductName', 1:'ModelName'})
    #print('Product Names DF:') #Check
    #display(df) #Check
    
    ##Read in info on bioproducts and merge with model dataframe (created above); merge based on product names
    BP_df = pd.read_excel(inputDirectory_general + 'Bioproducts_ModelInput.xlsx', sheet_name='Bioproducts_Info')
    print('Bioproduct DF: ')#Check
    #display(BP_df)#Check
    mergedBPdf = BP_df.merge(df, 'inner', 'ProductName') #Merge bioproduct info and model names
    print('Merged bioproduct DF:') #Check
    #display(mergedBPdf)#Check
        
    return(mergedBPdf)
#Check:
#productInfo(inputDirectory_models, inputDirectory_general)


#def substratePWinfo(substrates, inputDirectory_general, setPath):
def substratePWinfo(inputDirectory_general):  
    
    #Read assimilation pathways into dataframe
    if modelID is 'iHN637': 
        fileName = 'SubstrateUptakePW_Reactions_iHN637.xlsx'
    else: fileName = 'SubstrateUptakePW_Reactions_iJO1366.xlsx'
    pathwayRxnsDF = pd.read_excel(inputDirectory_general + '/' + fileName, sheet_name='Pathways')
    
    #Add "stop" column after last reaction column, as indicator to stop
    idx = pathwayRxnsDF.columns.get_loc("ReactionF")#Locate last reaction column
    pathwayRxnsDF.insert(loc=idx+1, column='Stop', value='Stop')#Insert stop indicator after last reaction column

    #Set compound names as index; inplace and drop needed to indicate change should be made to existing dataframe        
    pathwayRxnsDF.set_index('Compound', inplace=True, drop=True) 
    
    return(pathwayRxnsDF)
#Check:
#substrates = ['Acetate', 'Ethanol', 'EG_glycolate']
#substratePWinfo(substrates, inputDirectory_general)